In [ ]:
# from google.colab import drive
# drive.mount("/content/drive/")

## Setup Environment

In [ ]:
# Important: cd into the spm folder
# %cd spm

In [ ]:
# # --- Optional: Download and Install Miniconda ---
# !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
# !bash Miniconda3-latest-Linux-x86_64.sh -bfp /usr/local
# !conda init bash
# !conda config --set auto_update_conda false
# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
# !conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
!conda create -n spm python=3.10 --yes
!conda run -n spm pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!conda run -n spm pip install xformers
!conda run -n spm pip install -r requirements.txt

## Create Configs

### Step 1: Choose Base Model

Examples of `pretrained_sd_model`:
- SD v1.4: `CompVis/stable-diffusion-v1-4`
- SD v1.5: `runwayml/stable-diffusion-v1-5`
- SD v2.1: `stabilityai/stable-diffusion-2-1-base`
- WD1.5 beta3: `Birchlabs/wd-1-5-beta3-unofficial`
- SDXL: `stabilityai/stable-diffusion-xl-base-1.0`

If base model is v2.x, set `is_v2_model` to `true`.

In [ ]:
pretrained_sd_model = "CompVis/stable-diffusion-v1-4"  #@param {type: "string"}
is_v2_model = "false" #@param ["true", "false"]
is_v_prediction_model = "false" #@param ["true", "false"]

### Step 2: Choose Concept

- `target_concept`: Targeted concept for erasing
- `surrogate_concept`: Surrogate concept for defining model generation after erasure, empty string by default

In [ ]:
target_concept = 'Chesapeake Bay Retriever'  #@param {type: "string"}
surrogate_concept = ''  #@param {type: "string"}

### Step 3: SPM Settings

- `mode`: `erase_with_la` or `erase` (for ablation)
- `dim`: default to 1, other values for ablation
- `sampling_batch_size`: indicates how many latent anchors are sampled for each iteration, default to 4, can be reduced if there's not enough VRAM
- `la_strength`: indicates the latent anchoring loss strength that balancing the erasure and preservation, default to 1000 for SD v1.4 models, should be further tuned for other base models for better performance


In [ ]:
mode = 'erase_with_la' #@param ["erase_with_la", "erase"]
dim = 1 #@param {type: "number"}
sampling_batch_size = 4 #@param {type: "number"}
la_strength = 1000 #@param {type: "number"}
erasing_scale = 1.0 #@param {type: "number"}

### Step 4: Training Settings

There are two parts for training settings, SD settings and optimization settings.
Notice that `resolution` is set to 512 for SD v1.x, 768 for SD v2.x, and 1024 for SDXL.

In [ ]:
resolution = 512 #@param [512, 640, 768, 896, 960, 1024]
max_denoising_steps = 30  #@param {type: "number"}
dynamic_resolution = "true" #@param ["true", "false"]
clip_skip = 1 #@param [1, 2]

batch_size = 1  #@param {type: "number"}
iterations = 3000  #@param {type: "number"}
lr = 1e-4  #@param {type: "number"}
optimizer = "AdamW8bit" #@param {type: "string"}
lr_scheduler = "cosine_with_restarts" #@param {type: "string"}
lr_warmup_steps = 500 #@param {type: "number"}
lr_scheduler_num_cycles = 3 #@param {type: "number"}
save_per_steps = 500  #@param {type: "number"}
precision = "float32" #@param ["float32", "float16", "bfloat16"]
verbose = "false" #@param ["true", "false"]

### Step 5: (Optional) Tracking Training Details with WandB

You can setup your wandb token to track the training details, including training statistics (e.g. losses, learning rates) and visualizations.
Your wandb token can be retrieved from https://wandb.ai/authorize .

In [ ]:
wandb_token = "" #@param {type: "string"}

prompts_to_visualize = ["dog", "mickey", "woman"]  #@param {type: "string"}
generate_num = 2  #@param {type: "number"}

# track target & surrogate by default
prompts_to_visualize = [target_concept, surrogate_concept] + prompts_to_visualize

# login with your wandb token
if wandb_token != "":
    !wandb login {wandb_token}


### Step 6: Generate Config Files

Run the following code block and the config files are automatically generated.

In [ ]:
# you can custom these strings to distiguish your different exps
exp_name = target_concept.replace(" ", "_")
if surrogate_concept:
  exp_name += f"_to_{surrogate_concept.replace(' ', '_')}"
save_name = f"{exp_name}"
run_name = f"{exp_name}"

config_file_path = f"configs/{save_name}/config.yaml"
prompts_file_path = f"configs/{save_name}/prompt.yaml"


config_file_content = f"""
prompts_file: "{prompts_file_path}"

pretrained_model:
  name_or_path: "{pretrained_sd_model}"
  v2: {is_v2_model}
  v_pred: {is_v_prediction_model}
  clip_skip: {clip_skip}

network:
  rank: {dim}
  alpha: 1.0

train:
  precision: {precision}
  noise_scheduler: "ddim"
  iterations: {iterations}
  batch_size: {batch_size}
  lr: {lr}
  unet_lr: {lr}
  text_encoder_lr: {0.5 * lr}
  optimizer_type: "{optimizer}"
  lr_scheduler: "{lr_scheduler}"
  lr_warmup_steps: 500
  lr_scheduler_num_cycles: 3
  max_denoising_steps: {max_denoising_steps}

save:
  name: "{save_name}"
  path: "output/{save_name}"
  per_steps: {save_per_steps}
  precision: {precision}

logging:
  use_wandb: {"true" if wandb_token != "" else "false"}
  interval: 500
  seed: 0
  generate_num: {generate_num}
  run_name: "{run_name}"
  verbose: {verbose}
  prompts: {prompts_to_visualize}

other:
  use_xformers: true
"""

import os
if not os.path.exists(f"./configs/{save_name}"):
  os.makedirs(f"./configs/{save_name}")

with open(config_file_path, "w") as f:
  f.write(config_file_content)

prompts_file_content = f"""
- target: "{target_concept}"
  positive: "{target_concept}"
  unconditional: ""
  neutral: "{surrogate_concept}"
  action: "{mode}"
  guidance_scale: "{erasing_scale}"
  resolution: {resolution}
  batch_size: {batch_size}
  dynamic_resolution: {dynamic_resolution}
  la_strength: {la_strength}
  sampling_batch_size: {sampling_batch_size}
"""

with open(prompts_file_path, "w") as f:
  f.write(prompts_file_content)


## Start Training

In [ ]:
%%time
!conda run -n spm python ./train_spm.py --config_file configs/Chesapeake_Bay_Retriever/config.yaml

## Inference

In [ ]:
%%time
!conda run -n spm python infer_with_spm_from_csv.py \
  --csv_path prompts/spm_generate_chesapeake.csv \
  --spm_path output/Chesapeake_Bay_Retriever/ \
  --config configs/generation_chesapeake.yaml \
  --base_model CompVis/stable-diffusion-v1-4

In [ ]:
%%time
!conda run -n spm python infer_with_spm_from_csv.py \
  --csv_path prompts/spm_generate_bluetick.csv \
  --spm_path output/bluetick/ \
  --config configs/generation_bluetick.yaml \
  --base_model CompVis/stable-diffusion-v1-4